# LA Studio llm-chat — Qwen3.5 2B

This notebook loads exactly `qwen3.5-2b` (`Qwen/Qwen3.5-2B`) on CUDA.
It is independent from API Gateway and rejects every other model ID.

1. Choose **Runtime → Change runtime type → GPU**.
2. Run all cells.
3. Copy the printed URL and token into the matching LA Studio feature.


In [ ]:
!nvidia-smi
%pip install -q "fastapi==0.115.12" "uvicorn==0.34.3" "transformers>=5.6.0,<6" "accelerate>=1.12,<2" "safetensors>=0.6,<1"


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_chat_worker.py')
WORKER.write_text('import json\nimport os\nimport secrets\nimport threading\n\nimport torch\nfrom fastapi import Depends, FastAPI, Header, HTTPException, Request\nfrom fastapi.responses import StreamingResponse\nfrom pydantic import BaseModel, Field\nfrom transformers import (\n    AutoModelForMultimodalLM,\n    AutoProcessor,\n    StoppingCriteria,\n    StoppingCriteriaList,\n    TextIteratorStreamer,\n)\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. Choose a Colab GPU runtime; CPU fallback is disabled.")\n\nMODEL_ID = "qwen3.5-2b"\nMODEL_NAME = "Qwen3.5 2B"\nUPSTREAM_MODEL = "Qwen/Qwen3.5-2B"\nUPSTREAM_REVISION = "15852e8c16360a2fea060d615a32b45270f8a8fc"\nTOKEN = os.environ["LA_STUDIO_COLAB_CHAT_TOKEN"]\nMAX_CHAT_MESSAGES = 64\nMAX_CHAT_CHARS = 50000\nMAX_CHAT_TOKENS = 32768\nINFERENCE_SLOTS = threading.BoundedSemaphore(1)\n\nPROCESSOR = AutoProcessor.from_pretrained(UPSTREAM_MODEL, revision=UPSTREAM_REVISION)\nMODEL = AutoModelForMultimodalLM.from_pretrained(\n    UPSTREAM_MODEL,\n    revision=UPSTREAM_REVISION,\n    torch_dtype=torch.float16,\n    device_map={"": 0},\n    low_cpu_mem_usage=True,\n).eval()\nTOKENIZER = PROCESSOR.tokenizer\n\ndef authorize(authorization: str = Header(default="")):\n    if not secrets.compare_digest(authorization, "Bearer " + TOKEN):\n        raise HTTPException(status_code=401, detail="invalid or missing bearer token")\n\ndef require_exact_model(requested: str) -> None:\n    if requested.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{requested}\'. Open the notebook for the selected model.",\n        )\n\nclass ChatMessage(BaseModel):\n    role: str = Field(pattern="^(system|user|assistant)$")\n    content: str = Field(min_length=1, max_length=8000)\n\nclass ChatRequest(BaseModel):\n    model: str\n    messages: list[ChatMessage]\n    stream: bool = True\n    max_tokens: int = Field(default=1024, ge=1, le=MAX_CHAT_TOKENS)\n    context_tokens: int = Field(default=4096, ge=512, le=131072)\n    temperature: float = Field(default=0.7, ge=0.01, le=2.0)\n    top_p: float = Field(default=0.8, ge=0.01, le=1.0)\n    top_k: int = Field(default=20, ge=1, le=200)\n    repeat_penalty: float = Field(default=1.05, ge=0.8, le=2.0)\n\nclass DisconnectStop(StoppingCriteria):\n    def __init__(self, cancelled: threading.Event):\n        self.cancelled = cancelled\n    def __call__(self, input_ids, scores, **kwargs):\n        return self.cancelled.is_set()\n\napp = FastAPI(title="LA Studio Chat - Qwen3.5 2B", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(_: None = Depends(authorize)):\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "upstream_model": UPSTREAM_MODEL,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(_: None = Depends(authorize)):\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "llm-chat",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "upstream_model": UPSTREAM_MODEL,\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v1/chat/completions")\nasync def chat(request: ChatRequest, http_request: Request, _: None = Depends(authorize)):\n    require_exact_model(request.model)\n    if not request.stream:\n        raise HTTPException(status_code=400, detail="this direct worker requires stream=true")\n    if not request.messages:\n        raise HTTPException(status_code=400, detail="messages must not be empty")\n    if len(request.messages) > MAX_CHAT_MESSAGES or sum(len(item.content) for item in request.messages) > MAX_CHAT_CHARS:\n        raise HTTPException(status_code=413, detail="chat request is too large")\n    if not INFERENCE_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="worker is busy; retry shortly")\n    try:\n        messages = [{"role": item.role, "content": item.content} for item in request.messages]\n        inputs = PROCESSOR.apply_chat_template(\n            messages,\n            add_generation_prompt=True,\n            tokenize=True,\n            return_dict=True,\n            return_tensors="pt",\n            truncation=True,\n            max_length=request.context_tokens,\n        ).to("cuda")\n        streamer = TextIteratorStreamer(TOKENIZER, skip_prompt=True, skip_special_tokens=True)\n        cancelled = threading.Event()\n        generation_errors = []\n        def generate():\n            try:\n                MODEL.generate(\n                    **inputs,\n                    streamer=streamer,\n                    max_new_tokens=request.max_tokens,\n                    do_sample=True,\n                    temperature=request.temperature,\n                    top_p=request.top_p,\n                    top_k=request.top_k,\n                    repetition_penalty=request.repeat_penalty,\n                    pad_token_id=TOKENIZER.eos_token_id,\n                    stopping_criteria=StoppingCriteriaList([DisconnectStop(cancelled)]),\n                )\n            except Exception as error:\n                generation_errors.append(error)\n                streamer.end()\n        generation = threading.Thread(target=generate, daemon=True)\n        generation.start()\n    except Exception:\n        INFERENCE_SLOTS.release()\n        raise\n\n    async def events():\n        try:\n            for token in streamer:\n                if await http_request.is_disconnected():\n                    cancelled.set()\n                    break\n                yield "data: " + json.dumps({"choices": [{"delta": {"content": token}}]}, ensure_ascii=False) + "\\n\\n"\n            generation.join(timeout=10)\n            if generation_errors:\n                message = f"{type(generation_errors[0]).__name__}: {str(generation_errors[0])[:300]}"\n                yield "data: " + json.dumps({"error": {"message": message}}) + "\\n\\n"\n            yield "data: [DONE]\\n\\n"\n        finally:\n            cancelled.set()\n            INFERENCE_SLOTS.release()\n    return StreamingResponse(events(), media_type="text/event-stream", headers={"Cache-Control": "no-store"})' + '\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
MODEL_ID = 'qwen3.5-2b'

import json
import os
import secrets
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

TOKEN = secrets.token_urlsafe(32)
env = os.environ.copy()
env['LA_STUDIO_COLAB_CHAT_TOKEN'] = TOKEN
log_path = '/content/la_studio_chat_worker.log'
worker = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", 'la_studio_chat_worker:app', "--host", "127.0.0.1", "--port", '3944'],
    cwd="/content",
    env=env,
    stdout=open(log_path, "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(180):
    try:
        request = urllib.request.Request(
            "http://127.0.0.1:3944/health",
            headers={"Authorization": "Bearer " + TOKEN},
        )
        with urllib.request.urlopen(request, timeout=5) as response:
            health = json.load(response)
        if health.get("ready") and health.get("device") == "cuda" and health.get("model") == MODEL_ID:
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Exact-model worker did not become ready. Log tail:\n" + Path(log_path).read_text(errors="replace")[-5000:])

subprocess.run(
    ["wget", "-q", "-O", "/content/cloudflared.deb",
     "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"],
    check=True,
)
subprocess.run(["dpkg", "-i", "/content/cloudflared.deb"], check=True)
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3944", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
public_url = None
for _ in range(180):
    line = tunnel.stdout.readline()
    print(line, end="")
    if "https://" in line and "trycloudflare.com" in line:
        public_url = line[line.find("https://"):].split()[0]
        break
if not public_url:
    raise RuntimeError("Cloudflare tunnel URL was not found")

print("\nLA_STUDIO_COLAB_CHAT_URL=" + public_url)
print("LA_STUDIO_COLAB_CHAT_TOKEN=" + TOKEN)
print("LA_STUDIO_COLAB_CHAT_MODEL=" + MODEL_ID)
print("Paste only these direct-worker values into the matching LA Studio feature. Do not add /v1.")
